# 04. Exploratory Functional Lateralization Analysis

This notebook evaluates left/right hemisphere summaries and lateralization indices for layer-wise SRM+LLM encoding. These analyses are exploratory follow-ups to the main layer-wise encoding results.

**GitHub-ready notebook:** outputs have been cleared and private paths have been replaced with placeholders.

# Functional Lateralization Analysis

This notebook starts a focused hemisphere-level analysis for the SNL layer-wise encoding pipeline. The first step builds left/right hemisphere tables from the already generated voxel-wise layer maps. It does not refit SRM, GPT-2 features, or ridge models.


## Step 11A: Build Hemisphere-Specific Layer-Wise Tables

This step reads existing raw+LLM, SRM+LLM, and SRM-minus-raw voxel-wise encoding maps for each network and layer. Voxels are assigned to the left or right hemisphere using the original Schaefer parcel labels (`LH` / `RH`) after resampling the Schaefer atlas to the same BOLD-space grid used by the brain maps. The output is a set of hemisphere-specific layer-wise tables for later lateralization-index analysis.


In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import nibabel as nib
from nilearn import image

pd.set_option("display.float_format", lambda x: f"{x:.6f}")
np.set_printoptions(precision=6, suppress=True)

# =========================
# Project setup
# =========================

BASE_DIR = Path(r"YOUR_SNL2026_ROOT")
LAYER_ROOT = BASE_DIR / "Layer_wise_snl_only"
ENC_PROJECT_DIR = LAYER_ROOT / "SNL_layerwise_encoding_fixed_srm50"
SRM_RUNS_DIR = BASE_DIR / "runs"
STEP10_DIR = ENC_PROJECT_DIR / "step10_layerwise_brain_mapping"
STEP10_NII_DIR = STEP10_DIR / "nii"

OUT_DIR = LAYER_ROOT / "LRbrain"
STEP11A_DIR = OUT_DIR / "step11A_hemisphere_specific_layerwise_tables"
CSV_DIR = STEP11A_DIR / "csv"
JSON_DIR = STEP11A_DIR / "json"
for d in [OUT_DIR, STEP11A_DIR, CSV_DIR, JSON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NETWORK_NAMES = [
    "VisCent", "VisPeri",
    "SomMotA", "SomMotB",
    "DorsAttnA", "DorsAttnB",
    "SalVentAttnA", "SalVentAttnB",
    "LimbicA", "LimbicB",
    "ContA", "ContB", "ContC",
    "DefaultA", "DefaultB", "DefaultC",
    "TempPar",
]
LAYER_INDICES = list(range(0, 49))
MAP_TYPES = {
    "raw": "raw_group_encoding_r",
    "srm": "srm_group_encoding_r",
    "delta": "delta_group_encoding_r_srm_minus_raw",
}

# Full Schaefer atlas used for parcel IDs. The binary selected ROI mask is not enough for LH/RH labels.
ATLAS_PATH = Path(r"YOUR_SCHAEFER400_ATLAS_NIFTI_PATH")
if not ATLAS_PATH.exists():
    raise FileNotFoundError(f"Missing Schaefer atlas: {ATLAS_PATH}")

print("Functional lateralization Step 11A")
print(f"Encoding project directory: {ENC_PROJECT_DIR}")
print(f"Step10 NIfTI directory     : {STEP10_NII_DIR}")
print(f"Output directory           : {STEP11A_DIR}")
print(f"Atlas path                 : {ATLAS_PATH}")

start_time = time.time()


In [ ]:
# =========================
# Helper functions
# =========================

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def infer_hemisphere_from_label(label):
    label = str(label)
    if "_LH_" in label or label.startswith("LH_") or "-LH-" in label:
        return "left"
    if "_RH_" in label or label.startswith("RH_") or "-RH-" in label:
        return "right"
    return "unknown"


def robust_mean(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.mean(x))


def robust_median(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.median(x))


def robust_std(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    if x.size <= 1:
        return np.nan
    return float(np.std(x, ddof=1))


def summarize_values(values):
    values = np.asarray(values, dtype=np.float64)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return {
            "n_voxels": 0,
            "mean_r": np.nan,
            "median_r": np.nan,
            "std_r": np.nan,
            "min_r": np.nan,
            "max_r": np.nan,
            "positive_voxel_fraction": np.nan,
        }
    return {
        "n_voxels": int(finite.size),
        "mean_r": float(np.mean(finite)),
        "median_r": float(np.median(finite)),
        "std_r": float(np.std(finite, ddof=1)) if finite.size > 1 else np.nan,
        "min_r": float(np.min(finite)),
        "max_r": float(np.max(finite)),
        "positive_voxel_fraction": float(np.mean(finite > 0)),
    }


def load_selected_parcel_labels(roi_name):
    csv_path = SRM_RUNS_DIR / f"{roi_name}_400_parcels_fixed_srm50" / "step1_setup_roi_cleaned" / "csv" / "selected_parcel_labels.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing selected parcel label table for {roi_name}: {csv_path}")
    df = pd.read_csv(csv_path)
    if "parcel_id" not in df.columns or "parcel_name" not in df.columns:
        raise ValueError(f"Unexpected columns in {csv_path}: {df.columns.tolist()}")
    df["parcel_id"] = df["parcel_id"].astype(int)
    df["hemisphere"] = df["parcel_name"].apply(infer_hemisphere_from_label)
    return df


def build_roi_hemisphere_masks(roi_name, ref_img):
    """Return boolean masks for LH/RH voxels in the already-resampled BOLD/native grid."""
    label_df = load_selected_parcel_labels(roi_name)
    parcel_to_hemi = dict(zip(label_df["parcel_id"].astype(int), label_df["hemisphere"]))
    selected_ids = set(parcel_to_hemi.keys())

    atlas_nii = nib.load(str(ATLAS_PATH))
    atlas_resampled = image.resample_to_img(
        atlas_nii,
        ref_img,
        interpolation="nearest",
        force_resample=True,
        copy_header=True,
    )
    atlas_data = np.rint(atlas_resampled.get_fdata()).astype(np.int32)

    selected_mask = np.isin(atlas_data, list(selected_ids))
    left_mask = np.zeros(atlas_data.shape, dtype=bool)
    right_mask = np.zeros(atlas_data.shape, dtype=bool)
    unknown_mask = np.zeros(atlas_data.shape, dtype=bool)

    for parcel_id, hemi in parcel_to_hemi.items():
        parcel_mask = atlas_data == int(parcel_id)
        if hemi == "left":
            left_mask |= parcel_mask
        elif hemi == "right":
            right_mask |= parcel_mask
        else:
            unknown_mask |= parcel_mask

    return {
        "label_df": label_df,
        "selected_mask": selected_mask,
        "left_mask": left_mask,
        "right_mask": right_mask,
        "unknown_mask": unknown_mask,
        "atlas_data": atlas_data,
    }


def layer_nifti_path(roi_name, layer_index, suffix):
    return STEP10_NII_DIR / roi_name / f"{roi_name}_layer_{layer_index:02d}_{suffix}.nii.gz"


In [ ]:
# =========================
# Build hemisphere-specific layer-wise tables
# =========================

summary_rows = []
mask_rows = []
parcel_rows = []
missing_files = []

for roi_name in NETWORK_NAMES:
    roi_start = time.time()
    roi_nii_dir = STEP10_NII_DIR / roi_name
    if not roi_nii_dir.exists():
        raise FileNotFoundError(f"Missing Step10 NIfTI directory for {roi_name}: {roi_nii_dir}")

    # Use the first raw map as the reference image for this ROI's native/BOLD grid.
    ref_path = layer_nifti_path(roi_name, 0, MAP_TYPES["raw"])
    if not ref_path.exists():
        raise FileNotFoundError(f"Missing reference map for {roi_name}: {ref_path}")
    ref_img = nib.load(str(ref_path))

    hemi_info = build_roi_hemisphere_masks(roi_name, ref_img)
    left_mask = hemi_info["left_mask"]
    right_mask = hemi_info["right_mask"]
    unknown_mask = hemi_info["unknown_mask"]
    selected_mask = hemi_info["selected_mask"]
    label_df = hemi_info["label_df"].copy()
    label_df.insert(0, "roi_name", roi_name)
    parcel_rows.append(label_df)

    mask_rows.append({
        "roi_name": roi_name,
        "n_selected_parcels": int(label_df.shape[0]),
        "n_left_parcels": int((label_df["hemisphere"] == "left").sum()),
        "n_right_parcels": int((label_df["hemisphere"] == "right").sum()),
        "n_unknown_parcels": int((label_df["hemisphere"] == "unknown").sum()),
        "n_selected_voxels_from_atlas": int(selected_mask.sum()),
        "n_left_voxels_from_atlas": int(left_mask.sum()),
        "n_right_voxels_from_atlas": int(right_mask.sum()),
        "n_unknown_voxels_from_atlas": int(unknown_mask.sum()),
    })

    for layer_index in LAYER_INDICES:
        layer_values = {}
        layer_valid_masks = []
        for map_key, suffix in MAP_TYPES.items():
            nii_path = layer_nifti_path(roi_name, layer_index, suffix)
            if not nii_path.exists():
                missing_files.append(str(nii_path))
                continue
            data = np.asarray(nib.load(str(nii_path)).get_fdata(), dtype=np.float32)
            layer_values[map_key] = data
            layer_valid_masks.append(np.isfinite(data) & (data != 0))

        if len(layer_values) != len(MAP_TYPES):
            continue

        # The final valid ROI mask is the atlas-selected ROI intersected with nonzero finite map voxels.
        # This avoids including background zeros while preserving the Schaefer LH/RH parcel assignment.
        valid_map_mask = np.logical_or.reduce(layer_valid_masks)
        final_roi_mask = selected_mask & valid_map_mask

        for hemi_name, hemi_mask in [("left", left_mask), ("right", right_mask)]:
            use_mask = final_roi_mask & hemi_mask
            raw_vals = layer_values["raw"][use_mask]
            srm_vals = layer_values["srm"][use_mask]
            delta_vals = layer_values["delta"][use_mask]

            raw_summary = summarize_values(raw_vals)
            srm_summary = summarize_values(srm_vals)
            delta_summary = summarize_values(delta_vals)

            summary_rows.append({
                "roi_name": roi_name,
                "layer_index": int(layer_index),
                "layer_label": "embedding" if layer_index == 0 else f"layer_{layer_index:02d}",
                "layer_type": "embedding" if layer_index == 0 else "transformer",
                "hemisphere": hemi_name,
                "n_voxels": int(use_mask.sum()),
                "raw_mean_r": raw_summary["mean_r"],
                "raw_median_r": raw_summary["median_r"],
                "raw_std_r": raw_summary["std_r"],
                "raw_min_r": raw_summary["min_r"],
                "raw_max_r": raw_summary["max_r"],
                "raw_positive_voxel_fraction": raw_summary["positive_voxel_fraction"],
                "srm_mean_r": srm_summary["mean_r"],
                "srm_median_r": srm_summary["median_r"],
                "srm_std_r": srm_summary["std_r"],
                "srm_min_r": srm_summary["min_r"],
                "srm_max_r": srm_summary["max_r"],
                "srm_positive_voxel_fraction": srm_summary["positive_voxel_fraction"],
                "delta_mean_r_srm_minus_raw": delta_summary["mean_r"],
                "delta_median_r_srm_minus_raw": delta_summary["median_r"],
                "delta_std_r_srm_minus_raw": delta_summary["std_r"],
                "delta_min_r_srm_minus_raw": delta_summary["min_r"],
                "delta_max_r_srm_minus_raw": delta_summary["max_r"],
                "delta_positive_voxel_fraction": delta_summary["positive_voxel_fraction"],
            })

    print(f"Finished {roi_name:14s} in {(time.time() - roi_start) / 60:.2f} min")

if missing_files:
    raise FileNotFoundError("Missing Step10 layer maps:\n" + "\n".join(missing_files[:20]))

hemisphere_layerwise_df = pd.DataFrame(summary_rows)
mask_summary_df = pd.DataFrame(mask_rows)
parcel_hemisphere_df = pd.concat(parcel_rows, ignore_index=True)

# Basic sanity checks.
expected_rows = len(NETWORK_NAMES) * len(LAYER_INDICES) * 2
print("\nStep 11A completed")
print(f"Expected hemisphere rows: {expected_rows}")
print(f"Observed hemisphere rows: {len(hemisphere_layerwise_df)}")
print(f"Mask summary rows      : {len(mask_summary_df)}")
print(f"Parcel label rows      : {len(parcel_hemisphere_df)}")

if len(hemisphere_layerwise_df) != expected_rows:
    raise ValueError("Unexpected number of hemisphere-layer rows. Please inspect missing inputs or masks.")
if (hemisphere_layerwise_df["n_voxels"] <= 0).any():
    bad = hemisphere_layerwise_df[hemisphere_layerwise_df["n_voxels"] <= 0].head(20)
    raise ValueError(f"Some hemisphere-layer cells have zero voxels. First rows:\n{bad}")

# Save outputs. CSV uses fixed decimal formatting to avoid scientific notation.
hemisphere_csv = CSV_DIR / "step11A_hemisphere_layerwise_encoding_table.csv"
mask_csv = CSV_DIR / "step11A_hemisphere_mask_summary.csv"
parcel_csv = CSV_DIR / "step11A_schaefer_parcel_hemisphere_labels_used.csv"

hemisphere_layerwise_df.to_csv(hemisphere_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
mask_summary_df.to_csv(mask_csv, index=False, encoding="utf-8-sig", float_format="%.6f")
parcel_hemisphere_df.to_csv(parcel_csv, index=False, encoding="utf-8-sig", float_format="%.6f")

# Optional Excel workbook for easier inspection.
xlsx_path = CSV_DIR / "step11A_hemisphere_layerwise_tables.xlsx"
try:
    with pd.ExcelWriter(xlsx_path) as writer:
        hemisphere_layerwise_df.to_excel(writer, sheet_name="hemisphere_layerwise", index=False)
        mask_summary_df.to_excel(writer, sheet_name="mask_summary", index=False)
        parcel_hemisphere_df.to_excel(writer, sheet_name="parcel_labels", index=False)
    xlsx_written = True
except Exception as exc:
    print(f"Excel output skipped because pandas ExcelWriter failed: {exc}")
    xlsx_written = False

summary = {
    "analysis": "functional_lateralization_step11A",
    "hemisphere_definition": "Primary: Schaefer parcel label contains LH/RH after resampling full Schaefer atlas to the Step10 brain-map reference grid.",
    "base_dir": str(BASE_DIR),
    "encoding_project_dir": str(ENC_PROJECT_DIR),
    "step10_dir": str(STEP10_DIR),
    "atlas_path": str(ATLAS_PATH),
    "n_networks": int(len(NETWORK_NAMES)),
    "n_layers": int(len(LAYER_INDICES)),
    "n_rows_hemisphere_layerwise": int(len(hemisphere_layerwise_df)),
    "outputs": {
        "hemisphere_layerwise_csv": str(hemisphere_csv),
        "mask_summary_csv": str(mask_csv),
        "parcel_hemisphere_csv": str(parcel_csv),
        "xlsx": str(xlsx_path) if xlsx_written else None,
    },
    "elapsed_minutes": float((time.time() - start_time) / 60.0),
}
summary_json = JSON_DIR / "step11A_hemisphere_layerwise_summary.json"
with open(summary_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nSaved outputs:")
print(f"Hemisphere table : {hemisphere_csv}")
print(f"Mask summary     : {mask_csv}")
print(f"Parcel labels    : {parcel_csv}")
if xlsx_written:
    print(f"Excel workbook   : {xlsx_path}")
print(f"Summary JSON     : {summary_json}")
print(f"Elapsed minutes  : {summary['elapsed_minutes']:.2f}")

print("\nPreview:")
display(hemisphere_layerwise_df.head(12))
display(mask_summary_df)


## Step 11B: Compute Lateralization Index

This step converts the hemisphere-specific layer-wise table into lateralization-index tables. Positive LI values indicate left-biased encoding, and negative LI values indicate right-biased encoding. The analysis is computed separately for Raw+LLM, SRM+LLM, and SRM-minus-Raw gain.


In [ ]:
# Step 11B: Compute Lateralization Index
#
# This cell converts the left/right hemisphere summaries from Step 11A into
# lateralization indices (LI). Positive LI means stronger left-hemisphere values,
# and negative LI means stronger right-hemisphere values.
#
# To avoid inflated LI values when both hemispheres have near-zero encoding,
# LI is set to NaN when the denominator is smaller than LI_DENOM_MIN. The
# denominator and voxel-count imbalance are saved so lateralization results can
# be interpreted together with signal strength and hemisphere size imbalance.

from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FormatStrFormatter

BASE_DIR = Path(r"YOUR_PROJECT_ROOT")
PROJECT_DIR = BASE_DIR / "SNL_layerwise_encoding_fixed_srm50"
LR_ROOT = BASE_DIR / "LRbrain"
STEP11A_DIR = LR_ROOT / "step11A_hemisphere_specific_layerwise_tables"
STEP11B_DIR = LR_ROOT / "step11B_lateralization_index_differentenominator_thresholds"

CSV_DIR = STEP11B_DIR / "csv"
FIG_DIR = STEP11B_DIR / "figures"
JSON_DIR = STEP11B_DIR / "json"
XLSX_DIR = STEP11B_DIR / "xlsx"
for d in [STEP11B_DIR, CSV_DIR, FIG_DIR, JSON_DIR, XLSX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HEMI_TABLE_PATH = STEP11A_DIR / "csv" / "step11A_hemisphere_layerwise_encoding_table.csv"
if not HEMI_TABLE_PATH.exists():
    raise FileNotFoundError(f"Missing Step 11A hemisphere table: {HEMI_TABLE_PATH}")

start_time = time.time()
LI_DENOM_MIN = 0.05

hemi_df = pd.read_csv(HEMI_TABLE_PATH)

# Step 11A has intentionally compact column names. Harmonize them here so
# Step 11B can use stable names without requiring Step 11A to be rerun.
if "layer_name" not in hemi_df.columns and "layer_label" in hemi_df.columns:
    hemi_df["layer_name"] = hemi_df["layer_label"]

if "delta_mean_r" not in hemi_df.columns:
    if "delta_mean_r_srm_minus_raw" in hemi_df.columns:
        hemi_df["delta_mean_r"] = hemi_df["delta_mean_r_srm_minus_raw"]
    else:
        hemi_df["delta_mean_r"] = hemi_df["srm_mean_r"] - hemi_df["raw_mean_r"]

if "delta_median_r" not in hemi_df.columns:
    if "delta_median_r_srm_minus_raw" in hemi_df.columns:
        hemi_df["delta_median_r"] = hemi_df["delta_median_r_srm_minus_raw"]
    else:
        hemi_df["delta_median_r"] = hemi_df["srm_median_r"] - hemi_df["raw_median_r"]

network_family_map = {
    "VisCent": "Visual",
    "VisPeri": "Visual",
    "SomMotA": "SomMot",
    "SomMotB": "SomMot",
    "DorsAttnA": "DorsAttn",
    "DorsAttnB": "DorsAttn",
    "SalVentAttnA": "SalVentAttn",
    "SalVentAttnB": "SalVentAttn",
    "LimbicA": "Limbic",
    "LimbicB": "Limbic",
    "ContA": "Control",
    "ContB": "Control",
    "ContC": "Control",
    "DefaultA": "Default",
    "DefaultB": "Default",
    "DefaultC": "Default",
    "TempPar": "TempPar",
}
if "network_family" not in hemi_df.columns:
    hemi_df["network_family"] = hemi_df["roi_name"].map(network_family_map).fillna(hemi_df["roi_name"])

required_cols = [
    "roi_name", "network_family", "layer_index", "layer_name", "hemisphere",
    "raw_mean_r", "srm_mean_r", "delta_mean_r",
    "raw_median_r", "srm_median_r", "delta_median_r",
    "n_voxels",
]
missing_cols = [c for c in required_cols if c not in hemi_df.columns]
if missing_cols:
    raise ValueError(f"Step 11A table is missing required columns: {missing_cols}")

print("Loaded Step 11A hemisphere table")
print(f"Rows: {len(hemi_df)}")
print(f"LI denominator minimum: {LI_DENOM_MIN:.5f}")

index_cols = ["roi_name", "network_family", "layer_index", "layer_name"]
value_cols = [
    "raw_mean_r", "srm_mean_r", "delta_mean_r",
    "raw_median_r", "srm_median_r", "delta_median_r",
    "n_voxels",
]

wide_df = hemi_df.pivot_table(
    index=index_cols,
    columns="hemisphere",
    values=value_cols,
    aggfunc="first",
).reset_index()
wide_df.columns = [
    "_".join([str(x) for x in col if str(x) != ""]).strip("_")
    if isinstance(col, tuple) else col
    for col in wide_df.columns
]

needed_wide = []
for base in value_cols:
    needed_wide.extend([f"{base}_left", f"{base}_right"])
missing_wide = [c for c in needed_wide if c not in wide_df.columns]
if missing_wide:
    raise ValueError(f"Could not build left/right wide table; missing columns: {missing_wide}")

li_df = wide_df.copy()
li_df = li_df.rename(columns={
    "n_voxels_left": "n_voxels_left",
    "n_voxels_right": "n_voxels_right",
})

li_df["n_voxels_total"] = li_df["n_voxels_left"] + li_df["n_voxels_right"]
li_df["voxel_count_imbalance"] = (
    (li_df["n_voxels_left"] - li_df["n_voxels_right"]) / li_df["n_voxels_total"]
)


def lateralization_index(left, right, min_denom=LI_DENOM_MIN):
    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)
    denom = np.abs(left) + np.abs(right)
    li = np.full(denom.shape, np.nan, dtype=np.float64)
    valid = np.isfinite(left) & np.isfinite(right) & np.isfinite(denom) & (denom >= min_denom)
    li[valid] = (left[valid] - right[valid]) / denom[valid]
    return li, denom


def bias_direction(li_values):
    li_values = np.asarray(li_values, dtype=np.float64)
    return np.where(
        np.isnan(li_values),
        "undefined",
        np.where(li_values >= 0, "left", "right"),
    )

for metric in ["raw_mean_r", "srm_mean_r", "delta_mean_r", "raw_median_r", "srm_median_r", "delta_median_r"]:
    li_col = f"LI_{metric}"
    denom_col = f"{li_col}_denom"
    li_df[li_col], li_df[denom_col] = lateralization_index(
        li_df[f"{metric}_left"].values,
        li_df[f"{metric}_right"].values,
    )
    li_df[f"{li_col}_bias"] = bias_direction(li_df[li_col].values)

# A compact family-level summary, useful for checking whether lateralization is
# concentrated in larger functional families rather than one subdivision only.
network_family_li_df = (
    li_df
    .groupby(["network_family", "layer_index", "layer_name"], as_index=False)
    .agg(
        mean_LI_raw_mean_r=("LI_raw_mean_r", "mean"),
        mean_LI_srm_mean_r=("LI_srm_mean_r", "mean"),
        mean_LI_delta_mean_r=("LI_delta_mean_r", "mean"),
        mean_raw_LI_denom=("LI_raw_mean_r_denom", "mean"),
        mean_srm_LI_denom=("LI_srm_mean_r_denom", "mean"),
        mean_delta_LI_denom=("LI_delta_mean_r_denom", "mean"),
        mean_voxel_count_imbalance=("voxel_count_imbalance", "mean"),
        n_networks=("roi_name", "nunique"),
    )
)

# Select strongest absolute LI for each network and metric. Rows with undefined
# LI are ignored so near-zero denominator cases do not become artificial peaks.
strongest_rows = []
for roi_name, sub in li_df.groupby("roi_name"):
    for metric in ["LI_raw_mean_r", "LI_srm_mean_r", "LI_delta_mean_r"]:
        valid_sub = sub[np.isfinite(sub[metric])].copy()
        if valid_sub.empty:
            strongest_rows.append({
                "roi_name": roi_name,
                "metric": metric,
                "layer_index": np.nan,
                "layer_name": None,
                "LI": np.nan,
                "abs_LI": np.nan,
                "bias": "undefined",
                "LI_denom": np.nan,
                "n_voxels_left": float(sub["n_voxels_left"].iloc[0]),
                "n_voxels_right": float(sub["n_voxels_right"].iloc[0]),
                "n_voxels_total": float(sub["n_voxels_total"].iloc[0]),
                "voxel_count_imbalance": float(sub["voxel_count_imbalance"].iloc[0]),
            })
            continue

        idx = valid_sub[metric].abs().idxmax()
        row = valid_sub.loc[idx]
        strongest_rows.append({
            "roi_name": roi_name,
            "metric": metric,
            "layer_index": int(row["layer_index"]),
            "layer_name": row["layer_name"],
            "LI": float(row[metric]),
            "abs_LI": float(abs(row[metric])),
            "bias": row[f"{metric}_bias"],
            "LI_denom": float(row[f"{metric}_denom"]),
            "n_voxels_left": int(row["n_voxels_left"]),
            "n_voxels_right": int(row["n_voxels_right"]),
            "n_voxels_total": int(row["n_voxels_total"]),
            "voxel_count_imbalance": float(row["voxel_count_imbalance"]),
        })
strongest_li_df = pd.DataFrame(strongest_rows)

# Save tables.
li_csv = CSV_DIR / "step11B_lateralization_index_by_network_layer.csv"
family_csv = CSV_DIR / "step11B_lateralization_index_by_network_layer_family.csv"
strongest_csv = CSV_DIR / "step11B_strongest_lateralization_layers_by_network.csv"
li_df.to_csv(li_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
network_family_li_df.to_csv(family_csv, index=False, encoding="utf-8-sig", float_format="%.5f")
strongest_li_df.to_csv(strongest_csv, index=False, encoding="utf-8-sig", float_format="%.5f")

try:
    xlsx_path = XLSX_DIR / "step11B_lateralization_index_tables.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        li_df.to_excel(writer, sheet_name="network_layer_LI", index=False)
        network_family_li_df.to_excel(writer, sheet_name="family_layer_LI", index=False)
        strongest_li_df.to_excel(writer, sheet_name="strongest_LI", index=False)
except Exception as exc:
    print(f"Excel export skipped: {exc}")

# Figures: layer-wise LI profiles for raw, SRM, and delta.
sns.set_theme(style="whitegrid", context="talk")
metric_plot_specs = [
    ("LI_raw_mean_r", "Raw + LLM lateralization index", "layerwise_LI_raw_mean_r.png"),
    ("LI_srm_mean_r", "SRM + LLM lateralization index", "layerwise_LI_srm_mean_r.png"),
    ("LI_delta_mean_r", "SRM-related gain lateralization index", "layerwise_LI_delta_mean_r.png"),
]

for metric, title, filename in metric_plot_specs:
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.lineplot(
        data=li_df,
        x="layer_index",
        y=metric,
        hue="roi_name",
        linewidth=1.8,
        alpha=0.85,
        ax=ax,
    )
    ax.axhline(0, color="black", linewidth=1.0, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("GPT-2 representation index (0 = embedding; 1-48 = transformer layers)")
    ax.set_ylabel("LI = (left - right) / (|left| + |right|)")
    ax.set_ylim(-1.05, 1.05)
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    ax.legend(title="Network", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9)
    plt.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=260, bbox_inches="tight")
    plt.close(fig)

# Figure: voxel-count imbalance by network, because LI interpretation can be
# affected by strongly asymmetric ROI sizes.
imbalance_df = li_df[["roi_name", "network_family", "n_voxels_left", "n_voxels_right", "n_voxels_total", "voxel_count_imbalance"]].drop_duplicates()
fig, ax = plt.subplots(figsize=(12, 5.5))
sns.barplot(
    data=imbalance_df.sort_values("voxel_count_imbalance"),
    x="voxel_count_imbalance",
    y="roi_name",
    color="#4C72B0",
    ax=ax,
)
ax.axvline(0, color="black", linewidth=1.0)
ax.set_title("Left-right voxel-count imbalance by network")
ax.set_xlabel("(left voxels - right voxels) / total voxels")
ax.set_ylabel("Network")
ax.xaxis.set_major_formatter(FormatStrFormatter("%.2f"))
plt.tight_layout()
fig.savefig(FIG_DIR / "voxel_count_imbalance_by_network.png", dpi=260, bbox_inches="tight")
plt.close(fig)

summary = {
    "step": "step11B_lateralization_index",
    "input_table": str(HEMI_TABLE_PATH),
    "li_denominator_minimum": float(LI_DENOM_MIN),
    "n_network_layer_rows": int(len(li_df)),
    "n_family_layer_rows": int(len(network_family_li_df)),
    "n_nan_LI_raw_mean_r": int(li_df["LI_raw_mean_r"].isna().sum()),
    "n_nan_LI_srm_mean_r": int(li_df["LI_srm_mean_r"].isna().sum()),
    "n_nan_LI_delta_mean_r": int(li_df["LI_delta_mean_r"].isna().sum()),
    "mean_abs_LI_raw_mean_r": float(np.nanmean(np.abs(li_df["LI_raw_mean_r"]))),
    "mean_abs_LI_srm_mean_r": float(np.nanmean(np.abs(li_df["LI_srm_mean_r"]))),
    "mean_abs_LI_delta_mean_r": float(np.nanmean(np.abs(li_df["LI_delta_mean_r"]))),
    "max_abs_voxel_count_imbalance": float(np.nanmax(np.abs(li_df["voxel_count_imbalance"]))),
    "outputs": {
        "network_layer_li_csv": str(li_csv),
        "family_layer_li_csv": str(family_csv),
        "strongest_li_csv": str(strongest_csv),
        "figure_dir": str(FIG_DIR),
    },
    "elapsed_minutes": float((time.time() - start_time) / 60.0),
}

with open(JSON_DIR / "step11B_lateralization_index_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nStep 11B completed: lateralization index tables and figures")
print(f"Saved network-layer LI table: {li_csv}")
print(f"Saved family-layer LI table : {family_csv}")
print(f"Saved strongest LI table    : {strongest_csv}")
print("\nNaN LI counts after denominator filtering:")
print(f"raw mean r  : {summary['n_nan_LI_raw_mean_r']}")
print(f"SRM mean r  : {summary['n_nan_LI_srm_mean_r']}")
print(f"delta mean r: {summary['n_nan_LI_delta_mean_r']}")
print("\nMean absolute LI:")
print(f"Raw mean r  : {summary['mean_abs_LI_raw_mean_r']:.5f}")
print(f"SRM mean r  : {summary['mean_abs_LI_srm_mean_r']:.5f}")
print(f"Delta mean r: {summary['mean_abs_LI_delta_mean_r']:.5f}")
display(strongest_li_df.head(20))


In [ ]:
# ============================================================================
# Step 11C: Lateralization Validation Checks
# ============================================================================
# This step validates the hemisphere lateralization analysis in three ways.
# First, it repeats the LI calculation across several denominator thresholds
# to test whether the lateralization pattern depends on one arbitrary cutoff.
# Second, it saves the original left/right values and sign patterns so that LI
# can be interpreted correctly, especially for SRM-minus-Raw changes. Third, it
# summarizes LI by log-scaled GPT-2 layer-depth bins rather than relying only on
# the single strongest layer.

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter

pd.set_option("display.float_format", lambda x: f"{x:.5f}")
warnings.filterwarnings("ignore", category=RuntimeWarning)

start_time = time.time()

# -------------------------
# Project paths
# -------------------------
BASE_DIR = Path(r"YOUR_PROJECT_ROOT")
LR_ROOT = BASE_DIR / "LRbrain"
STEP11A_DIR = LR_ROOT / "step11A_hemisphere_specific_layerwise_tables"
STEP11C_DIR = LR_ROOT / "step11C_lateralization_validation_checks"
CSV_DIR = STEP11C_DIR / "csv"
JSON_DIR = STEP11C_DIR / "json"
FIG_DIR = STEP11C_DIR / "figures"
XLSX_DIR = STEP11C_DIR / "xlsx"

for d in [STEP11C_DIR, CSV_DIR, JSON_DIR, FIG_DIR, XLSX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STEP11A_TABLE = STEP11A_DIR / "csv" / "step11A_hemisphere_layerwise_encoding_table.csv"
if not STEP11A_TABLE.exists():
    raise FileNotFoundError(f"Missing Step 11A table: {STEP11A_TABLE}")

print("Step 11C: Lateralization validation checks")
print(f"Input table: {STEP11A_TABLE}")
print(f"Output directory: {STEP11C_DIR}")

# -------------------------
# Settings
# -------------------------
LI_DENOM_MIN_LIST = [0.02, 0.03, 0.05, 0.08]
PRIMARY_THRESHOLD = 0.05
KEY_NETWORKS = ["TempPar", "DefaultA", "DefaultB", "SalVentAttnB", "SomMotB"]

METRIC_SPECS = [
    {
        "metric": "raw_mean_r",
        "left_col": "raw_mean_r_left",
        "right_col": "raw_mean_r_right",
        "label": "Raw+LLM mean r",
    },
    {
        "metric": "srm_mean_r",
        "left_col": "srm_mean_r_left",
        "right_col": "srm_mean_r_right",
        "label": "SRM+LLM mean r",
    },
    {
        "metric": "delta_mean_r_srm_minus_raw",
        "left_col": "delta_mean_r_srm_minus_raw_left",
        "right_col": "delta_mean_r_srm_minus_raw_right",
        "label": "SRM-minus-Raw mean r",
    },
    {
        "metric": "raw_median_r",
        "left_col": "raw_median_r_left",
        "right_col": "raw_median_r_right",
        "label": "Raw+LLM median r",
    },
    {
        "metric": "srm_median_r",
        "left_col": "srm_median_r_left",
        "right_col": "srm_median_r_right",
        "label": "SRM+LLM median r",
    },
    {
        "metric": "delta_median_r_srm_minus_raw",
        "left_col": "delta_median_r_srm_minus_raw_left",
        "right_col": "delta_median_r_srm_minus_raw_right",
        "label": "SRM-minus-Raw median r",
    },
]

NETWORK_FAMILY_MAP = {
    "VisCent": "Visual",
    "VisPeri": "Visual",
    "SomMotA": "SomMot",
    "SomMotB": "SomMot",
    "DorsAttnA": "DorsAttn",
    "DorsAttnB": "DorsAttn",
    "SalVentAttnA": "SalVentAttn",
    "SalVentAttnB": "SalVentAttn",
    "ContA": "Control",
    "ContB": "Control",
    "ContC": "Control",
    "DefaultA": "Default",
    "DefaultB": "Default",
    "DefaultC": "Default",
    "TempPar": "TempPar",
    "LimbicA": "Limbic",
    "LimbicB": "Limbic",
}

NETWORK_ORDER = [
    "VisCent", "VisPeri", "SomMotA", "SomMotB", "DorsAttnA", "DorsAttnB",
    "SalVentAttnA", "SalVentAttnB", "LimbicA", "LimbicB",
    "ContA", "ContB", "ContC", "DefaultA", "DefaultB", "DefaultC", "TempPar",
]

LAYER_BIN_ORDER = ["embedding", "early", "early_middle", "middle_late", "late"]

# -------------------------
# Helper functions
# -------------------------
def fmt_float(x, digits=5):
    if x is None or not np.isfinite(x):
        return "NA"
    return f"{float(x):.{digits}f}"


def log_depth_percent(layer_index, max_transformer_layer=48):
    layer_index = int(layer_index)
    if layer_index <= 0:
        return 0.0
    return float(np.log1p(layer_index) / np.log1p(max_transformer_layer) * 100.0)


def layer_depth_bin(layer_index):
    layer_index = int(layer_index)
    if layer_index == 0:
        return "embedding"
    depth = log_depth_percent(layer_index)
    if depth < 50:
        return "early"
    if depth < 72:
        return "early_middle"
    if depth < 90:
        return "middle_late"
    return "late"


def sign_pattern(left_value, right_value, eps=1e-8):
    if not (np.isfinite(left_value) and np.isfinite(right_value)):
        return "nonfinite"

    left_sign = 0 if abs(left_value) <= eps else (1 if left_value > 0 else -1)
    right_sign = 0 if abs(right_value) <= eps else (1 if right_value > 0 else -1)

    if left_sign == 0 and right_sign == 0:
        return "both_near_zero"
    if left_sign > 0 and right_sign > 0:
        return "both_positive"
    if left_sign < 0 and right_sign < 0:
        return "both_negative"
    if left_sign > 0 and right_sign < 0:
        return "left_positive_right_negative"
    if left_sign < 0 and right_sign > 0:
        return "left_negative_right_positive"
    if left_sign > 0 and right_sign == 0:
        return "left_positive_right_near_zero"
    if left_sign == 0 and right_sign > 0:
        return "left_near_zero_right_positive"
    if left_sign < 0 and right_sign == 0:
        return "left_negative_right_near_zero"
    if left_sign == 0 and right_sign < 0:
        return "left_near_zero_right_negative"
    return "other"


def compute_li_row(left_value, right_value, threshold):
    if not (np.isfinite(left_value) and np.isfinite(right_value)):
        return np.nan, np.nan, "undefined", "nonfinite"

    denom = abs(float(left_value)) + abs(float(right_value))
    pattern = sign_pattern(float(left_value), float(right_value))

    if denom < threshold:
        return np.nan, denom, "undefined", pattern

    li = (float(left_value) - float(right_value)) / denom
    if abs(li) <= 1e-12:
        bias = "balanced"
    elif li > 0:
        bias = "left"
    else:
        bias = "right"
    return float(li), float(denom), bias, pattern


def safe_fraction(mask):
    mask = pd.Series(mask)
    if len(mask) == 0:
        return np.nan
    return float(mask.mean())


def save_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8-sig", float_format="%.5f")
    print(f"Saved: {path}")


def format_axis_decimal(ax):
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.5f"))
    ax.yaxis.set_major_formatter(FormatStrFormatter("%.5f"))
    ax.xaxis.offsetText.set_visible(False)
    ax.yaxis.offsetText.set_visible(False)

# -------------------------
# Load and reshape Step 11A table
# -------------------------
hemi_df = pd.read_csv(STEP11A_TABLE)
required_cols = [
    "roi_name", "layer_index", "layer_label", "layer_type", "hemisphere", "n_voxels",
    "raw_mean_r", "raw_median_r", "srm_mean_r", "srm_median_r",
    "delta_mean_r_srm_minus_raw", "delta_median_r_srm_minus_raw",
]
missing_cols = [c for c in required_cols if c not in hemi_df.columns]
if missing_cols:
    raise ValueError(f"Step 11A table is missing required columns: {missing_cols}")

hemi_df = hemi_df.copy()
for col in [c for c in required_cols if c not in ["roi_name", "layer_label", "layer_type", "hemisphere"]]:
    hemi_df[col] = pd.to_numeric(hemi_df[col], errors="coerce")

hemi_df["network_family"] = hemi_df["roi_name"].map(NETWORK_FAMILY_MAP).fillna("Other")
hemi_df["log_layer_depth_percent"] = hemi_df["layer_index"].map(log_depth_percent)
hemi_df["layer_bin"] = hemi_df["layer_index"].map(layer_depth_bin)

# Step 11A may store hemisphere labels as either left/right or LH/RH.
# Normalize the labels here so Step 11C is robust to either convention.
hemi_label = hemi_df["hemisphere"].astype(str).str.strip().str.lower()
left_df = hemi_df[hemi_label.isin(["left", "lh", "l"])].copy()
right_df = hemi_df[hemi_label.isin(["right", "rh", "r"])].copy()

if left_df.empty or right_df.empty:
    unique_hemi = sorted(hemi_df["hemisphere"].astype(str).unique().tolist())
    raise ValueError(
        "Could not identify both left and right hemisphere rows. "
        f"Observed hemisphere labels: {unique_hemi}"
    )

merge_keys = ["roi_name", "network_family", "layer_index", "layer_label", "layer_type", "log_layer_depth_percent", "layer_bin"]
value_cols = [
    "n_voxels", "raw_mean_r", "raw_median_r", "srm_mean_r", "srm_median_r",
    "delta_mean_r_srm_minus_raw", "delta_median_r_srm_minus_raw",
]

wide_df = left_df[merge_keys + value_cols].merge(
    right_df[merge_keys + value_cols],
    on=merge_keys,
    how="inner",
    suffixes=("_left", "_right"),
)

wide_df["n_voxels_total"] = wide_df["n_voxels_left"] + wide_df["n_voxels_right"]
wide_df["voxel_count_imbalance"] = (
    wide_df["n_voxels_left"] - wide_df["n_voxels_right"]
) / wide_df["n_voxels_total"]

if wide_df.empty:
    raise ValueError("No matched LH/RH rows were found after reshaping Step 11A table.")

print("Loaded Step 11A hemisphere table")
print(f"Input rows: {len(hemi_df)}")
print(f"Matched network-layer rows: {len(wide_df)}")
print(f"Networks: {wide_df['roi_name'].nunique()}")
print(f"Layer indices: {int(wide_df['layer_index'].min())} to {int(wide_df['layer_index'].max())}")

# -------------------------
# Validation 1 and 2: threshold sensitivity plus original left/right values
# -------------------------
li_rows = []
for threshold in LI_DENOM_MIN_LIST:
    for _, row in wide_df.iterrows():
        for spec in METRIC_SPECS:
            left_value = float(row[spec["left_col"]])
            right_value = float(row[spec["right_col"]])
            li, denom, bias, pattern = compute_li_row(left_value, right_value, threshold)
            li_rows.append({
                "threshold": float(threshold),
                "roi_name": row["roi_name"],
                "network_family": row["network_family"],
                "layer_index": int(row["layer_index"]),
                "layer_label": row["layer_label"],
                "layer_type": row["layer_type"],
                "log_layer_depth_percent": float(row["log_layer_depth_percent"]),
                "layer_bin": row["layer_bin"],
                "metric": spec["metric"],
                "metric_label": spec["label"],
                "left_value": left_value,
                "right_value": right_value,
                "left_minus_right": left_value - right_value,
                "LI_denom": denom,
                "LI": li,
                "abs_LI": abs(li) if np.isfinite(li) else np.nan,
                "bias": bias,
                "sign_pattern": pattern,
                "n_voxels_left": int(row["n_voxels_left"]),
                "n_voxels_right": int(row["n_voxels_right"]),
                "n_voxels_total": int(row["n_voxels_total"]),
                "voxel_count_imbalance": float(row["voxel_count_imbalance"]),
            })

li_df = pd.DataFrame(li_rows)
li_df["threshold_label"] = li_df["threshold"].map(lambda x: f"denom_min_{x:.2f}")
li_df["is_valid_LI"] = np.isfinite(li_df["LI"])
li_df["is_delta_metric"] = li_df["metric"].str.contains("delta", regex=False)

full_li_csv = CSV_DIR / "step11C_li_by_threshold_network_layer.csv"
save_csv(li_df, full_li_csv)

# -------------------------
# Threshold sensitivity summaries
# -------------------------
summary_rows = []
for (threshold, metric), g in li_df.groupby(["threshold", "metric"], sort=False):
    valid = g[g["is_valid_LI"]].copy()
    summary_rows.append({
        "threshold": float(threshold),
        "metric": metric,
        "metric_label": g["metric_label"].iloc[0],
        "total_n": int(len(g)),
        "valid_n": int(len(valid)),
        "undefined_n": int(len(g) - len(valid)),
        "valid_pct": float(len(valid) / len(g) * 100.0) if len(g) else np.nan,
        "mean_LI": float(valid["LI"].mean()) if len(valid) else np.nan,
        "median_LI": float(valid["LI"].median()) if len(valid) else np.nan,
        "mean_abs_LI": float(valid["abs_LI"].mean()) if len(valid) else np.nan,
        "median_abs_LI": float(valid["abs_LI"].median()) if len(valid) else np.nan,
        "left_bias_fraction": safe_fraction(valid["bias"].eq("left")) if len(valid) else np.nan,
        "right_bias_fraction": safe_fraction(valid["bias"].eq("right")) if len(valid) else np.nan,
        "balanced_fraction": safe_fraction(valid["bias"].eq("balanced")) if len(valid) else np.nan,
        "n_abs_ge_0_5": int((valid["abs_LI"] >= 0.5).sum()) if len(valid) else 0,
        "n_abs_eq_1": int(np.isclose(valid["abs_LI"], 1.0, atol=1e-8).sum()) if len(valid) else 0,
    })

overall_sensitivity_df = pd.DataFrame(summary_rows)
overall_sensitivity_csv = CSV_DIR / "step11C_threshold_sensitivity_overall.csv"
save_csv(overall_sensitivity_df, overall_sensitivity_csv)

network_summary_rows = []
for (threshold, metric, roi_name), g in li_df.groupby(["threshold", "metric", "roi_name"], sort=False):
    valid = g[g["is_valid_LI"]].copy()
    network_summary_rows.append({
        "threshold": float(threshold),
        "metric": metric,
        "metric_label": g["metric_label"].iloc[0],
        "roi_name": roi_name,
        "network_family": g["network_family"].iloc[0],
        "total_n": int(len(g)),
        "valid_n": int(len(valid)),
        "undefined_n": int(len(g) - len(valid)),
        "valid_pct": float(len(valid) / len(g) * 100.0) if len(g) else np.nan,
        "mean_LI": float(valid["LI"].mean()) if len(valid) else np.nan,
        "median_LI": float(valid["LI"].median()) if len(valid) else np.nan,
        "mean_abs_LI": float(valid["abs_LI"].mean()) if len(valid) else np.nan,
        "left_bias_fraction": safe_fraction(valid["bias"].eq("left")) if len(valid) else np.nan,
        "right_bias_fraction": safe_fraction(valid["bias"].eq("right")) if len(valid) else np.nan,
        "both_positive_fraction": safe_fraction(valid["sign_pattern"].eq("both_positive")) if len(valid) else np.nan,
        "both_negative_fraction": safe_fraction(valid["sign_pattern"].eq("both_negative")) if len(valid) else np.nan,
        "opposite_sign_fraction": safe_fraction(valid["sign_pattern"].isin(["left_positive_right_negative", "left_negative_right_positive"])) if len(valid) else np.nan,
        "mean_left_value": float(valid["left_value"].mean()) if len(valid) else np.nan,
        "mean_right_value": float(valid["right_value"].mean()) if len(valid) else np.nan,
        "mean_left_minus_right": float(valid["left_minus_right"].mean()) if len(valid) else np.nan,
    })

network_sensitivity_df = pd.DataFrame(network_summary_rows)
network_sensitivity_df["roi_name"] = pd.Categorical(network_sensitivity_df["roi_name"], categories=NETWORK_ORDER, ordered=True)
network_sensitivity_df = network_sensitivity_df.sort_values(["threshold", "metric", "roi_name"]).reset_index(drop=True)
network_sensitivity_csv = CSV_DIR / "step11C_threshold_sensitivity_by_network.csv"
save_csv(network_sensitivity_df, network_sensitivity_csv)

# Strongest valid LI layer for each threshold x metric x network.
strongest_rows = []
for (threshold, metric, roi_name), g in li_df.groupby(["threshold", "metric", "roi_name"], sort=False):
    valid = g[g["is_valid_LI"]].copy()
    if valid.empty:
        continue
    best_row = valid.loc[valid["abs_LI"].idxmax()].copy()
    strongest_rows.append(best_row.to_dict())

strongest_li_df = pd.DataFrame(strongest_rows)
if len(strongest_li_df):
    strongest_li_df["roi_name"] = pd.Categorical(strongest_li_df["roi_name"], categories=NETWORK_ORDER, ordered=True)
    strongest_li_df = strongest_li_df.sort_values(["threshold", "metric", "roi_name"]).reset_index(drop=True)
strongest_li_csv = CSV_DIR / "step11C_strongest_li_by_threshold_network.csv"
save_csv(strongest_li_df, strongest_li_csv)

# -------------------------
# Validation 3: layer-bin LI summaries
# -------------------------
layer_bin_rows = []
for (threshold, metric, roi_name, layer_bin), g in li_df.groupby(["threshold", "metric", "roi_name", "layer_bin"], sort=False):
    valid = g[g["is_valid_LI"]].copy()
    layer_bin_rows.append({
        "threshold": float(threshold),
        "metric": metric,
        "metric_label": g["metric_label"].iloc[0],
        "roi_name": roi_name,
        "network_family": g["network_family"].iloc[0],
        "layer_bin": layer_bin,
        "n_layers_in_bin": int(len(g)),
        "n_valid_layers": int(len(valid)),
        "n_undefined_layers": int(len(g) - len(valid)),
        "valid_pct": float(len(valid) / len(g) * 100.0) if len(g) else np.nan,
        "mean_LI": float(valid["LI"].mean()) if len(valid) else np.nan,
        "median_LI": float(valid["LI"].median()) if len(valid) else np.nan,
        "mean_abs_LI": float(valid["abs_LI"].mean()) if len(valid) else np.nan,
        "median_abs_LI": float(valid["abs_LI"].median()) if len(valid) else np.nan,
        "left_bias_fraction": safe_fraction(valid["bias"].eq("left")) if len(valid) else np.nan,
        "right_bias_fraction": safe_fraction(valid["bias"].eq("right")) if len(valid) else np.nan,
        "balanced_fraction": safe_fraction(valid["bias"].eq("balanced")) if len(valid) else np.nan,
        "both_positive_fraction": safe_fraction(valid["sign_pattern"].eq("both_positive")) if len(valid) else np.nan,
        "both_negative_fraction": safe_fraction(valid["sign_pattern"].eq("both_negative")) if len(valid) else np.nan,
        "opposite_sign_fraction": safe_fraction(valid["sign_pattern"].isin(["left_positive_right_negative", "left_negative_right_positive"])) if len(valid) else np.nan,
        "mean_left_value": float(valid["left_value"].mean()) if len(valid) else np.nan,
        "mean_right_value": float(valid["right_value"].mean()) if len(valid) else np.nan,
        "mean_left_minus_right": float(valid["left_minus_right"].mean()) if len(valid) else np.nan,
        "mean_LI_denom": float(valid["LI_denom"].mean()) if len(valid) else np.nan,
    })

layer_bin_summary_df = pd.DataFrame(layer_bin_rows)
layer_bin_summary_df["roi_name"] = pd.Categorical(layer_bin_summary_df["roi_name"], categories=NETWORK_ORDER, ordered=True)
layer_bin_summary_df["layer_bin"] = pd.Categorical(layer_bin_summary_df["layer_bin"], categories=LAYER_BIN_ORDER, ordered=True)
layer_bin_summary_df = layer_bin_summary_df.sort_values(["threshold", "metric", "roi_name", "layer_bin"]).reset_index(drop=True)
layer_bin_summary_csv = CSV_DIR / "step11C_layer_bin_li_summary.csv"
save_csv(layer_bin_summary_df, layer_bin_summary_csv)

key_network_summary_df = network_sensitivity_df[
    network_sensitivity_df["roi_name"].astype(str).isin(KEY_NETWORKS)
].copy()
key_network_summary_csv = CSV_DIR / "step11C_key_network_threshold_summary.csv"
save_csv(key_network_summary_df, key_network_summary_csv)

key_layer_bin_summary_df = layer_bin_summary_df[
    layer_bin_summary_df["roi_name"].astype(str).isin(KEY_NETWORKS)
].copy()
key_layer_bin_summary_csv = CSV_DIR / "step11C_key_network_layer_bin_summary.csv"
save_csv(key_layer_bin_summary_df, key_layer_bin_summary_csv)

# -------------------------
# Figures
# -------------------------
primary_metrics = ["raw_mean_r", "srm_mean_r", "delta_mean_r_srm_minus_raw"]
plot_overall_df = overall_sensitivity_df[overall_sensitivity_df["metric"].isin(primary_metrics)].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for metric, g in plot_overall_df.groupby("metric", sort=False):
    label = g["metric_label"].iloc[0]
    axes[0].plot(g["threshold"], g["valid_pct"], marker="o", linewidth=2, label=label)
    axes[1].plot(g["threshold"], g["mean_abs_LI"], marker="o", linewidth=2, label=label)

axes[0].set_title("LI valid-layer percentage by denominator threshold")
axes[0].set_xlabel("LI denominator threshold")
axes[0].set_ylabel("Valid layers (%)")
axes[0].set_ylim(0, 100)
axes[0].grid(alpha=0.25)

axes[1].set_title("Mean absolute LI by denominator threshold")
axes[1].set_xlabel("LI denominator threshold")
axes[1].set_ylabel("Mean absolute LI")
axes[1].grid(alpha=0.25)

for ax in axes:
    format_axis_decimal(ax)
axes[1].legend(frameon=False, bbox_to_anchor=(1.02, 1.0), loc="upper left")
plt.tight_layout()
threshold_fig = FIG_DIR / "step11C_threshold_sensitivity_overview.png"
plt.savefig(threshold_fig, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {threshold_fig}")

# Layer-bin summary bar plots at primary threshold.
primary_layer_df = layer_bin_summary_df[
    (np.isclose(layer_bin_summary_df["threshold"].astype(float), PRIMARY_THRESHOLD)) &
    (layer_bin_summary_df["metric"].isin(["srm_mean_r", "delta_mean_r_srm_minus_raw"]))
].copy()

for metric in ["srm_mean_r", "delta_mean_r_srm_minus_raw"]:
    metric_df = primary_layer_df[primary_layer_df["metric"].eq(metric)].copy()
    if metric_df.empty:
        continue
    pivot = metric_df.pivot(index="roi_name", columns="layer_bin", values="mean_LI")
    pivot = pivot.reindex(index=NETWORK_ORDER, columns=LAYER_BIN_ORDER)

    fig, ax = plt.subplots(figsize=(9.5, 6.5))
    im = ax.imshow(pivot.values.astype(float), aspect="auto", cmap="coolwarm", vmin=-0.5, vmax=0.5)
    ax.set_xticks(np.arange(len(LAYER_BIN_ORDER)))
    ax.set_xticklabels(LAYER_BIN_ORDER, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(f"Layer-bin LI summary: {metric_df['metric_label'].iloc[0]}")
    ax.set_xlabel("GPT-2 layer-depth bin")
    ax.set_ylabel("Network")
    cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label("Mean LI")
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
    cbar.ax.yaxis.offsetText.set_visible(False)
    plt.tight_layout()
    out_fig = FIG_DIR / f"step11C_layer_bin_mean_LI_{metric}.png"
    plt.savefig(out_fig, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out_fig}")

# Key-network sign-pattern table for delta metric at primary threshold.
primary_delta_key_df = li_df[
    (np.isclose(li_df["threshold"].astype(float), PRIMARY_THRESHOLD)) &
    (li_df["metric"].eq("delta_mean_r_srm_minus_raw")) &
    (li_df["roi_name"].isin(KEY_NETWORKS))
].copy()

if len(primary_delta_key_df):
    sign_counts = (
        primary_delta_key_df[primary_delta_key_df["is_valid_LI"]]
        .groupby(["roi_name", "sign_pattern"], observed=False)
        .size()
        .reset_index(name="n_layers")
        .sort_values(["roi_name", "n_layers"], ascending=[True, False])
    )
    save_csv(sign_counts, CSV_DIR / "step11C_primary_threshold_delta_sign_pattern_counts_key_networks.csv")

# -------------------------
# Optional Excel workbook
# -------------------------
xlsx_path = XLSX_DIR / "step11C_lateralization_validation_tables.xlsx"
try:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        overall_sensitivity_df.to_excel(writer, sheet_name="threshold_overall", index=False)
        network_sensitivity_df.to_excel(writer, sheet_name="threshold_by_network", index=False)
        strongest_li_df.to_excel(writer, sheet_name="strongest_li", index=False)
        layer_bin_summary_df.to_excel(writer, sheet_name="layer_bin_summary", index=False)
        key_network_summary_df.to_excel(writer, sheet_name="key_networks", index=False)
        key_layer_bin_summary_df.to_excel(writer, sheet_name="key_layer_bins", index=False)
    print(f"Saved: {xlsx_path}")
except Exception as exc:
    print(f"Excel export skipped: {exc}")

# -------------------------
# JSON summary and console readout
# -------------------------
summary = {
    "step": "Step 11C",
    "purpose": "Validate hemisphere lateralization index across denominator thresholds, retain left/right source values, and summarize LI by log-scaled GPT-2 layer-depth bins.",
    "input_table": str(STEP11A_TABLE),
    "output_dir": str(STEP11C_DIR),
    "thresholds": LI_DENOM_MIN_LIST,
    "primary_threshold": PRIMARY_THRESHOLD,
    "metrics": [spec["metric"] for spec in METRIC_SPECS],
    "n_matched_network_layer_rows": int(len(wide_df)),
    "n_li_rows": int(len(li_df)),
    "key_networks": KEY_NETWORKS,
    "layer_bins": {
        "embedding": "layer index 0",
        "early": "log depth < 50%",
        "middle": "50% <= log depth < 72%",
        "middle_late": "72% <= log depth < 90%",
        "late": "log depth >= 90%",
    },
    "elapsed_minutes": float((time.time() - start_time) / 60.0),
}

json_path = JSON_DIR / "step11C_lateralization_validation_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"Saved: {json_path}")

print("\nPrimary-threshold overview (mean metrics only):")
primary_overall = overall_sensitivity_df[
    (np.isclose(overall_sensitivity_df["threshold"].astype(float), PRIMARY_THRESHOLD)) &
    (overall_sensitivity_df["metric"].isin(primary_metrics))
][[
    "metric", "valid_n", "undefined_n", "valid_pct", "mean_LI", "mean_abs_LI",
    "left_bias_fraction", "right_bias_fraction", "n_abs_ge_0_5", "n_abs_eq_1"
]].copy()
display(primary_overall)

print("\nKey-network threshold sensitivity (primary threshold, delta mean r):")
primary_key_delta = key_network_summary_df[
    (np.isclose(key_network_summary_df["threshold"].astype(float), PRIMARY_THRESHOLD)) &
    (key_network_summary_df["metric"].eq("delta_mean_r_srm_minus_raw"))
][[
    "roi_name", "valid_n", "valid_pct", "mean_LI", "mean_abs_LI",
    "left_bias_fraction", "right_bias_fraction", "both_positive_fraction",
    "both_negative_fraction", "opposite_sign_fraction", "mean_left_value", "mean_right_value",
]].copy()
display(primary_key_delta)

print("\nStep 11C completed successfully.")
print(f"Elapsed: {(time.time() - start_time) / 60.0:.2f} min")
